# restated — retrieval over NVIDIA 10-K filings

**Research question:** does chunking strategy affect whether the system retrieves the *original* or the *restated* figure — and does reranking help or hurt?

The pipeline is the apparatus. The report is the deliverable.

*The Gen Academy — Mastering Agentic AI Bootcamp, Week 2, Project 2 (Track 2: LangChain + LangGraph)*

---

## Why this corpus

Three consecutive NVIDIA 10-Ks (FY2024, FY2025, FY2026) — one company across periods, rather than the more obvious many-companies-one-year design.

A cross-company corpus tests **attribution** (don't mix up AMD's number with Intel's). A single-company multi-period corpus tests **period confusion**, which is harder: every 10-K says "revenue increased" in near-identical language year over year, so the embeddings cannot separate the years and metadata has to do the work.

## 1 · Load keys

Two credentials. `NEBIUS_API_KEY` for embeddings and generation; `SEC_USER_AGENT` (a real name + email) because SEC blocks requests without one — the most common reason a fetch script silently fails.

Put them in `.env` (gitignored) or set them as environment variables.

In [1]:
import os, sys, json, time, subprocess
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass

for var in ("NEBIUS_API_KEY", "SEC_USER_AGENT"):
    val = os.environ.get(var, "")
    print(f"{var:18s} {'OK  (' + str(len(val)) + ' chars)' if val else 'MISSING'}")

print(f"\nproject root: {ROOT}")

NEBIUS_API_KEY     OK  (236 chars)
SEC_USER_AGENT     OK  (33 chars)

project root: C:\Users\joshitham_waybetterm\Documents\Restated


> **If `NEBIUS_API_KEY` says MISSING:** create a key at [tokenfactory.nebius.com](https://tokenfactory.nebius.com/project/api-keys) (Token Factory, *not* AI Cloud), then either put it in `.env` or run `$env:NEBIUS_API_KEY = "..."` and restart the kernel.

## 2 · See which models this account has

In [2]:
from openai import OpenAI

BASE_URL = "https://api.tokenfactory.nebius.com/v1/"
client = OpenAI(api_key=os.environ["NEBIUS_API_KEY"], base_url=BASE_URL)

models = sorted(m.id for m in client.models.list().data)
is_embed = lambda m: any(k in m.lower() for k in ("embed", "bge", "e5", "gte"))

print("EMBEDDING models:")
for m in models:
    if is_embed(m):
        print(" ", m)

print(f"\nCHAT models: {sum(1 for m in models if not is_embed(m))} available, e.g.")
for m in [m for m in models if not is_embed(m)][:5]:
    print(" ", m)

EMBEDDING models:
  Qwen/Qwen3-Embedding-8B

CHAT models: 23 available, e.g.
  MiniMaxAI/MiniMax-M3
  NousResearch/Hermes-4-405B
  Qwen/Qwen3-235B-A22B-Instruct-2507
  Qwen/Qwen3-30B-A3B-Instruct-2507
  Qwen/Qwen3.5-397B-A17B


## 3 · Choose the two models and test them

`check_embedding_ctx_length=False` is **required** for Nebius. By default `OpenAIEmbeddings` tokenizes locally and sends integer arrays, which OpenAI accepts but Nebius rejects with `400 {'detail': 'Tokenized input is not supported'}`.

In [3]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

EMBED_MODEL = "Qwen/Qwen3-Embedding-8B"
CHAT_MODEL  = "Qwen/Qwen3-235B-A22B-Instruct-2507"

embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL, base_url=BASE_URL,
    api_key=os.environ["NEBIUS_API_KEY"],
    check_embedding_ctx_length=False,   # <- required for Nebius
)
llm = ChatOpenAI(
    model=CHAT_MODEL, base_url=BASE_URL,
    api_key=os.environ["NEBIUS_API_KEY"],
    temperature=0.0, max_tokens=700,
)

v = embeddings.embed_query("Data Center revenue")
print(f"embedding dim: {len(v)}")
print("chat:", llm.invoke("Reply with exactly: ready").content)

embedding dim: 4096


chat: ready


## 4 · Download the filings

Ticker → CIK is resolved at runtime against SEC's official mapping rather than hardcoded. A wrong CIK fails in the worst way: it silently returns a *different company's* filings rather than erroring.

In [4]:
from fetch import download

filings = download("NVDA", limit=3)
for f in filings:
    print(f"{f.report_date}  filed {f.filing_date}  {f.accession}  {f.primary_doc}")

NVDA -> CIK 0001045810


  2026-01-25  filed 2026-02-25  0001045810-26-000021  nvda-20260125.htm
    exists, skipping (1,967,816 bytes)
  2025-01-26  filed 2025-02-26  0001045810-25-000023  nvda-20250126.htm
    exists, skipping (2,067,520 bytes)
  2024-01-28  filed 2024-02-21  0001045810-24-000029  nvda-20240128.htm
    exists, skipping (2,085,566 bytes)
2026-01-25  filed 2026-02-25  0001045810-26-000021  nvda-20260125.htm
2025-01-26  filed 2025-02-26  0001045810-25-000023  nvda-20250126.htm
2024-01-28  filed 2024-02-21  0001045810-24-000029  nvda-20240128.htm


Note each **filing date is ~1 month after its period end** (2026-02-25 vs 2026-01-25). Tagging `fiscal_year` from the filing date would shift every chunk into the wrong year — so it comes from inline XBRL instead.

## 5 · Clean the HTML

The only genuinely custom code in the pipeline. Everything downstream is stock LangChain.

Seven problems in EDGAR markup — five anticipated, two found only by inspecting the output:

1. `<table>` used for both financial data and page layout → layout tables dropped
2. Currency symbols in their own cells → `['Data Center', '$', '193,737', ...]` folded before alignment
3. Item section headings need tagging so answers can cite a section
4. Empty `<tr>` rows → dropped
5. Header and data rows have different cell counts (2 then 4) → align on the widest row
6. **Zero `<p>` tags.** Body text is `<div>` wrapping `<span>`. The first version produced 600 tables and *0 prose blocks*
7. **The hidden `<ix:header>`** holds ~19,000 chars of XBRL machine metadata. Left in, it became the single largest chunk in the corpus and was retrievable for any query

In [5]:
from clean import build_blocks, read_meta, RAW_DIR

raw_files = sorted(RAW_DIR.glob("nvda-*.htm")) + sorted(RAW_DIR.glob("nvda-*.html"))
seen, paths = set(), []
for p in raw_files:
    key = "".join(c for c in p.stem if c.isdigit())
    if key not in seen:
        seen.add(key); paths.append(p)

BLOCKS = {}
for path in paths:
    meta = read_meta(path)
    blocks = build_blocks(path)
    BLOCKS[meta.fiscal_year] = [b.__dict__ for b in blocks]
    n_tab = sum(1 for b in blocks if b.block_type == "table")
    print(f"FY{meta.fiscal_year}  {path.name:22s} {len(blocks):4d} blocks "
          f"({n_tab} table, {len(blocks)-n_tab} prose)")

FY2024  nvda-20240128.htm       665 blocks (58 table, 607 prose)


FY2025  nvda-20250126.htm       686 blocks (59 table, 627 prose)


FY2026  nvda-20260125.htm       689 blocks (56 table, 633 prose)


### Metadata comes from inline XBRL, never from prose

Regex over prose is *actively wrong* here. `for the fiscal year ended` matches the **comparative prior year** before the cover page, in all three filings:

| File | Naive prose match | Actual FY |
|---|---|---|
| `nvda-20260125.htm` | January 26, 2025 | **2026** |
| `nvda-20250126.htm` | January 28, 2024 | **2025** |
| `nvda-20240128.htm` | January 29, 2023 | **2024** |

`dei:DocumentFiscalYearFocus` gives the year exactly, with no inference.

In [6]:
# Look at a real extracted table. Do the numbers line up under their headers?
t = next(b for b in BLOCKS[2026] if b["block_type"] == "table" and "193,737" in b["text"])
print("citation:     ", t["citation"])
print("years_present:", t["years_present"])
print()
print(t["text"][:520])

citation:      FY2026, Item 15, "The following table summarizes revenue by specialized markets:"
years_present: [2026, 2025, 2024]

The following table summarizes revenue by specialized markets:
| Year Ended |  |  |  |
| --- | --- | --- | --- |
| Jan 25, 2026 | Jan 26, 2025 | Jan 28, 2024 |  |
| Revenue by End Market: | (In millions) |  |  |
| Data Center | 193,737 | 115,186 | 47,525 |
| Compute | 162,361 | 102,196 | 38,950 |
| Networking | 31,376 | 12,990 | 8,575 |
| Gaming | 16,042 | 11,350 | 10,447 |
| Professional Visualization | 3,191 | 1,878 | 1,553 |
| Automotive | 2,349 | 1,694 | 1,091 |
| OEM and Other | 619 | 389 | 306 |
| Total reven


`years_present` is a **list**, not a scalar — a held-whole table spans three fiscal years. Metadata-filtered retrieval must filter on `years_present`; filtering on a scalar `fiscal_year` would wrongly exclude every multi-year table.

## 6 · Chunk it — three ways · **graded comparison #1**

A whole filing won't fit in a context window, and you wouldn't want it to — more text means more to get confused by. So documents get cut into chunks.

- **Fixed** — cut every ~1000 characters with 150 overlap, so a sentence split across a boundary still appears whole somewhere. Completely blind to what the text says. This is `RecursiveCharacterTextSplitter`.
- **Semantic** — cut where the *topic* changes, measured by comparing the meaning of consecutive sentences. This is `SemanticChunker` at the 90th percentile. Chunk sizes vary, because topics do.
- **Structural** *(third arm, our own)* — cut on boundaries the document already declares: Item section, then heading, then a size budget. Never mid-paragraph.

One rule applies to all three: **tables are never split.** If only one strategy protected tables, you couldn't tell whether a difference came from where boundaries fall or from how tables were treated. Holding it constant is what makes the comparison mean anything.

A table that exceeds the budget is *still* not split — a half table is worse than a long chunk, because the header row carries the fiscal years and without it every figure is unattributable.

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from collections import defaultdict

ALL_BLOCKS = [b for fy in sorted(BLOCKS) for b in BLOCKS[fy]]
prose  = [b for b in ALL_BLOCKS if b["block_type"] == "prose"]
tables = [b for b in ALL_BLOCKS if b["block_type"] == "table"]

# Join paragraphs back into one document per (fiscal year, section).
#
# The cleaner deliberately emits small blocks so tables can be isolated and
# every block tagged with its Item section. But a chunker needs CONTINUOUS
# text: semantic chunking looks for the point where a topic shifts, and a lone
# paragraph has no such point. Handing it ~1,900 paragraphs would mean ~1,900
# round trips to the embedding model, each asking a question with no answer.
#
# Merging also keeps the comparison honest: all three strategies receive
# IDENTICAL input, so the only thing that differs is where boundaries fall.
buckets = defaultdict(list)
for b in prose:
    buckets[(b["fiscal_year"], b["item_section"])].append(b)

SECTIONS = [
    Document(page_content="\n\n".join(b["text"] for b in group),
             metadata=dict(group[0]))
    for group in buckets.values()
]

print(f"{len(prose):,} prose blocks -> {len(SECTIONS)} section documents")
print(f"{len(tables)} tables (kept whole by all three strategies)")

C:\Users\joshitham_waybetterm\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1,867 prose blocks -> 63 section documents
173 tables (kept whole by all three strategies)


In [8]:
fixed_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
FIXED = fixed_splitter.split_documents(SECTIONS)
print(f"fixed:      {len(FIXED) + len(tables):,} chunks  ({len(FIXED):,} prose + {len(tables)} tables)")

fixed:      1,446 chunks  (1,273 prose + 173 tables)


The next cell is the slow one — semantic chunking sends every sentence to the embedding model to find where meaning shifts. Expect a minute or two.

In [9]:
from langchain_experimental.text_splitter import SemanticChunker

t0 = time.time()
semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90,
)
SEMANTIC = semantic_splitter.split_documents(SECTIONS)
print(f"semantic:   {len(SEMANTIC) + len(tables):,} chunks  ({time.time()-t0:.0f}s)")

C:\Users\joshitham_waybetterm\AppData\Local\Temp\ipykernel_17692\2098482301.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


semantic:   723 chunks  (119s)


In [10]:
# Third arm: structural boundaries the document already declares.
from chunk import chunk_semantic

STRUCTURAL = []
for fy in sorted(BLOCKS):
    STRUCTURAL += [c for c in chunk_semantic(BLOCKS[fy], fy) if c.block_type == "prose"]
print(f"structural: {len(STRUCTURAL) + len(tables):,} chunks")

structural: 1,421 chunks


In [11]:
import statistics as st

arms = (
    ("fixed",      [d.page_content for d in FIXED]),
    ("semantic",   [d.page_content for d in SEMANTIC]),
    ("structural", [c.text for c in STRUCTURAL]),
)
for name, texts in arms:
    lens = [len(t) for t in texts]
    print(f"{name:11s} n={len(lens):5d}  chars: mean={st.mean(lens):6.0f} "
          f"median={st.median(lens):6.0f} min={min(lens):5d} max={max(lens):6d}")

print("\nFixed chunks are all about the same size — that is what 'fixed' means.")
print("Semantic sizes vary, because topics vary in length. Note the maximum:")
print("SemanticChunker has no size ceiling, so one chunk runs to ~19,000 chars —")
print("long enough to dilute its own embedding. That is a real finding, not a bug.")

fixed       n= 1273  chars: mean=   727 median=   808 min=   92 max=   999
semantic    n=  550  chars: mean=  1602 median=   924 min=    3 max= 19154
structural  n= 1248  chars: mean=   706 median=   777 min=   77 max=  1000

Fixed chunks are all about the same size — that is what 'fixed' means.
Semantic sizes vary, because topics vary in length. Note the maximum:
SemanticChunker has no size ceiling, so one chunk runs to ~19,000 chars —
long enough to dilute its own embedding. That is a real finding, not a bug.


### The control, asserted rather than assumed

`verify_chunks.py` checks five things before any indexing. The one that matters most: table text is **byte-identical** across strategies, not merely equal in count.

If a control fails, run A vs run B measures something other than chunking strategy and the headline result is void — cheaper to learn before spending embedding credits.

In [12]:
def run(script, *args):
    r = subprocess.run([sys.executable, str(ROOT / "src" / script), *args],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
    return r.returncode

run("chunk.py")          # writes data/chunks/{fixed,semantic}/
run("verify_chunks.py")  # asserts the experiment is validly controlled


=== fixed ===
  fixed     FY2024   478 chunks  (58 table, 420 prose)  prose chars: median 777, max 997
  fixed     FY2025   489 chunks  (59 table, 430 prose)  prose chars: median 794, max 999
  fixed     FY2026   483 chunks  (56 table, 427 prose)  prose chars: median 763, max 1000

=== semantic ===
  semantic  FY2024   469 chunks  (58 table, 411 prose)  prose chars: median 778, max 997
  semantic  FY2025   479 chunks  (59 table, 420 prose)  prose chars: median 787, max 1000
  semantic  FY2026   473 chunks  (56 table, 417 prose)  prose chars: median 760, max 1000

-> data\chunks/{fixed,semantic}/FY*.jsonl

=== control 1: tables identical across strategies ===
  FY2024: fixed=58 semantic=58 count_match=True text_identical=True
  FY2025: fixed=59 semantic=59 count_match=True text_identical=True
  FY2026: fixed=56 semantic=56 count_match=True text_identical=True

=== control 2: no table split (header row present in every table chunk) ===
  fixed    : 0 table chunks missing a header separa

0

## 7 · What the corpus actually contains

The interesting part was **not assumed** — it was found by scanning for line items whose prior-year figure changed between filings.

In [13]:
run("find_restatements2.py")

FY2026: 489 (topic, label, year) figures
FY2025: 559 (topic, label, year) figures
FY2024: 534 (topic, label, year) figures

=== RECLASSIFIED (33) ===
  [balance_sheet] 'Additional paid-in capital' FY2024
      FY2025: 13109  FY2024: 13132
  [balance_sheet] 'Other assets' FY2025
      FY2026: 3038  FY2025: 6425
  [deferred_taxes] 'Gross deferred tax assets' FY2024
      FY2025: 8060  FY2024: 8000
  [deferred_taxes] 'Other deferred tax assets' FY2025
      FY2026: 566  FY2025: 360
  [deferred_taxes] 'Property, equipment and intangible assets' FY2024
      FY2025: 64  FY2024: 4
  [deferred_taxes] 'Total deferred tax assets' FY2024
      FY2025: 6508  FY2024: 6448
  [eps_reconciliation] 'Basic (1)' FY2023
      FY2025: 0.18  FY2024: 1.76
  [eps_reconciliation] 'Diluted (2)' FY2023
      FY2025: 0.17  FY2024: 1.74
  [income_statement] 'Basic' FY2023
      FY2025: 0.18  FY2024: 1.76
  [income_statement] 'Diluted' FY2023
      FY2025: 0.17  FY2024: 1.74
  [income_statement] 'Interest income' 

0

### Trap 1 — reclassification

NVIDIA changed the basis of geographic revenue disaggregation between filings, and **says so in the table caption**:

- FY2025 filing: "based upon the **billing location of the customer**"
- FY2026 filing: "based upon the **location of the customers' headquarters**"

| Line item (FY2025) | Per FY2026 filing (HQ) | Per FY2025 filing (billing) |
|---|---|---|
| United States | 77,482 | 61,257 |
| China (incl. HK) | 25,048 | 17,108 |
| Other | 4,367 | 7,875 |

US and China rise while "Other" falls — what a billing→HQ reattribution looks like. Each figure appears in only one filing, so **which number the system returns is decided entirely by which chunk it retrieves**.

Notably, revenue *by end market* is **stable** (Data Center FY2025 is `115,186` in both). Had the eval set been written against revenue — the obvious choice — this category would have measured nothing.

### Trap 2 — the 10-for-1 stock split

NVIDIA split 10-for-1 in June 2024, so the FY2024 filing states per-share figures **pre-split** while later filings restate them:

| Line item (FY2024) | Per FY2025/26 | Per FY2024 |
|---|---|---|
| Basic EPS | 1.21 | 12.05 |
| Diluted weighted average shares | 24,940 | 2,494 |

Exactly 10x, unambiguous, both correct. A system answering "$12.05" is right about the number and **wrong about the basis** — only a citation reveals which.

### A mistake worth keeping in the record

An earlier scan reported **Taiwan FY2025 as `1,481` vs `20,573` — a 13.9x restatement**, written up as the headline finding.

It was wrong. The `1,481` comes from a *long-lived assets* table; the `20,573` from a *revenue* table. Both rows are labelled `Taiwan`.

The diagnostic had committed the exact error the pipeline is designed to avoid: **treating a line-item label as a unique key**. It is kept here because it is the clearest evidence for why `table_caption` is load-bearing — without it, a system asked "revenue attributed to Taiwan" would return a long-lived-assets figure.

## 8 · The eval set — written BEFORE retrieval was built

This ordering is the point. Writing questions after seeing what the pipeline retrieves well is how an eval set gets quietly tuned toward the system's existing strengths.

18 questions across six categories. Every expected figure is **verified against the filings** — an eval set with a wrong expected value scores a correct system as wrong, which is worse than no eval set.

In [14]:
import yaml
from collections import Counter

spec = yaml.safe_load((ROOT / "eval/questions.yaml").read_text(encoding="utf-8"))
QUESTIONS = spec["questions"]

print(f"{len(QUESTIONS)} questions\n")
for cat, n in Counter(q["category"] for q in QUESTIONS).most_common():
    print(f"  {cat:14s} {n}")

q = next(q for q in QUESTIONS if q["id"] == "q_restate_01")
print(f"\nthe restatement case:\n  {q['question']}\n  {q['expected']}")

19 questions

  numeric        4
  restatement    4
  temporal       3
  trend          3
  unanswerable   3
  split          2

the restatement case:
  What were NVIDIA's United States revenues in fiscal 2025?
  {'per_fy2026_filing': 77482, 'per_fy2025_filing': 61257}


In [15]:
run("verify_eval.py")

  [ok ] q_numeric_01  Data Center FY2026                       193,737  (x1 in FY2026 filing)
  [ok ] q_numeric_02  total revenue FY2026                     215,938  (x6 in FY2026 filing)
  [ok ] q_numeric_03  Networking FY2026                         31,376  (x1 in FY2026 filing)
  [ok ] q_numeric_04  Gaming FY2025 (in FY2026 filing)          11,350  (x1 in FY2026 filing)
  [ok ] q_numeric_04  Gaming FY2025 (in FY2025 filing)          11,350  (x1 in FY2025 filing)
  [ok ] q_temporal_01 Data Center FY2025 (in FY2026 filing)    115,186  (x1 in FY2026 filing)
  [ok ] q_temporal_02 total revenue FY2024 (in FY2026 filing)    60,922  (x4 in FY2026 filing)
  [ok ] q_temporal_02 total revenue FY2024 (in FY2024 filing)    60,922  (x6 in FY2024 filing)
  [ok ] q_temporal_03 total revenue FY2025 (in FY2025 filing)   130,497  (x6 in FY2025 filing)
  [ok ] q_trend_01    Data Center FY2024 (in FY2026 filing)     47,525  (x1 in FY2026 filing)
  [ok ] q_restate_01  US FY2025 per FY2026 filing        

0

## 9 · Put them in Chroma

One collection per strategy, so every arm is queryable without re-indexing. This is the slow, paid step — each chunk goes to the embedding model once.

Re-running is safe: chunk IDs are stable hashes, so a collection with a matching count is skipped unless `--force` is passed.

In [16]:
run("check_env.py")

=== packages ===
  ok    langchain                  1.4.0
  ok    langchain_openai           1.6.2
  ok    langchain_community        0.4.2
  ok    langchain_text_splitters   
  ok    chromadb                   1.5.9
  ok    rank_bm25                  
  ok    yaml                       6.0.3
  ok    pandas                     2.3.3
  ok    dotenv                     
  ok    sentence_transformers      6.0.1
  ok    langsmith                  0.8.3

=== corpus ===
  raw filings: 3
    nvda-20240128.htm  2,085,566 bytes
    nvda-20250126.htm  2,067,520 bytes
    nvda-20260125.htm  1,967,816 bytes

=== chunks ===
  fixed    : 1,450 chunks
  semantic : 1,421 chunks

=== credentials ===
  ok    NEBIUS_API_KEY set (236 chars, ends ...Kr8N)
  ok    SEC_USER_AGENT set

ready to index: 2,871 chunks across two strategies



0

In [17]:
# Smoke test first -- 20 chunks per strategy, to confirm credentials and
# batching work before committing the full corpus to embedding credits.
run("index.py", "--limit", "20", "--force")

model: Qwen/Qwen3-Embedding-8B
store: data\chroma
LIMIT: 20 chunks per strategy (smoke test)

  fixed: 20/20 embedded
  fixed: 20 vectors in 3.2s (6.2/s)      
  semantic: 20/20 embedded
  semantic: 20 vectors in 4.6s (4.4/s)      

40 vectors embedded in 7.8s
-> data\chroma



0

In [18]:
# Collections are already built. index.py skips a collection whose vector
# count matches, so re-running is a cheap no-op -- pass --force to rebuild.
run("index.py")

model: Qwen/Qwen3-Embedding-8B
store: data\chroma

  fixed: 64/1,450 embedded
  fixed: 128/1,450 embedded
  fixed: 192/1,450 embedded
  fixed: 256/1,450 embedded
  fixed: 320/1,450 embedded
  fixed: 384/1,450 embedded
  fixed: 448/1,450 embedded
  fixed: 512/1,450 embedded
  fixed: 576/1,450 embedded
  fixed: 640/1,450 embedded
  fixed: 704/1,450 embedded
  fixed: 768/1,450 embedded
  fixed: 832/1,450 embedded
  fixed: 896/1,450 embedded
  fixed: 960/1,450 embedded
  fixed: 1,024/1,450 embedded
  fixed: 1,088/1,450 embedded
  fixed: 1,152/1,450 embedded
  fixed: 1,216/1,450 embedded
  fixed: 1,280/1,450 embedded
  fixed: 1,344/1,450 embedded
  fixed: 1,408/1,450 embedded
  fixed: 1,450/1,450 embedded
  fixed: 1,450 vectors in 138.0s (10.5/s)      
  semantic: 64/1,421 embedded
  semantic: 128/1,421 embedded
  semantic: 192/1,421 embedded
  semantic: 256/1,421 embedded
  semantic: 320/1,421 embedded
  semantic: 384/1,421 embedded
  semantic: 448/1,421 embedded
  semantic: 512/1,421 embe

0

The geographic query returns all three filings' tables with the basis change visible in the captions — both sides of the restatement, retrieved together.

Period confusion is still visible in the numeric query: an FY2024 chunk appears for a fiscal-2026 question. Dense retrieval cannot reject it. That is what the metadata-filtered arm is for.

## 10 · Retrieval — and why we use two kinds

Dense (embeddings) finds passages that *mean* the same thing; BM25 finds passages that use the same *words*. Financial queries need both — "Data Center revenue" is a phrase to match literally, while "how fast is the business growing" is not.

Reciprocal rank fusion combines them. RRF uses only **rank**, not score, so no normalisation is needed between dense cosine distance and BM25's unbounded scores:

$$\text{score}(d) = \sum_{r} \frac{1}{k + \text{rank}_r(d)}$$

`k=60` is the value from the original RRF paper, used unchanged — tuning it on this corpus would be fitting the retriever to the eval set, which is exactly what the eval-before-retrieval rule exists to prevent.

In [20]:
from retrieve import build

Q = "What was NVIDIA's basic earnings per share in fiscal 2024?"

for kind in ("dense", "bm25", "hybrid", "filtered"):
    r = build(kind, "fixed", embeddings if kind != "bm25" else None)
    t0 = time.perf_counter()
    hits = r.search(Q, k=3)
    ms = (time.perf_counter() - t0) * 1000
    print(f"\n{kind:9s} [{ms:6.1f}ms]")
    for h in hits:
        print(f"   {h.score:7.4f} {h.block_type:6s} {h.citation[:64]}")


dense     [3304.3ms]
    0.6706 prose  FY2024, Item 7
    0.6684 prose  FY2024, Item 7
    0.6659 table  FY2024, Item 15, "NVIDIA Corporation and Subsidiaries"

bm25      [  13.6ms]
   22.7725 table  FY2026, Item 15, "The following is the basic and diluted net inc
   22.4310 table  FY2025, Item 15, "The following is a reconciliation of the denom
   22.1006 table  FY2024, Item 15, "The following is a reconciliation of the denom



hybrid    [ 585.7ms]
    0.0290 prose  FY2025, Item 15
    0.0287 table  FY2024, Item 15, "NVIDIA Corporation and Subsidiaries"
    0.0283 prose  FY2024, Item 15



filtered  [2944.2ms]
    0.6703 prose  FY2024, Item 7
    0.6683 prose  FY2024, Item 7
    0.6554 prose  FY2025, Item 7


Note the latency gap: BM25 answers in ~2ms, dense in ~200ms, because dense pays for an
embedding API call on every query.

### Is the answer even reachable?

A retrieval score of 0% can mean two very different things — the answer is not indexed, or
it is indexed but ranked below the cutoff. Those need opposite fixes, so it is worth
checking directly.

In [21]:
run("debug_reach.py")


dense / fixed   (searching to depth 50)
  q_numeric_01     rank 11  found '193,737'
  q_numeric_03     rank 12  found '31,376'
  q_numeric_04     rank 14  found '11,350'
  q_temporal_01    rank 18  found '193,737'
  q_restate_01     NOT in top 50
  q_restate_02     rank 36  found '25,048'
  q_split_01       NOT in top 50
  q_split_02       rank 31  found '2,494'

bm25 / fixed   (searching to depth 50)
  q_numeric_01     rank 20  found '193,737'
  q_numeric_03     rank 11  found '31,376'
  q_numeric_04     rank 16  found '11,350'
  q_temporal_01    rank 25  found '193,737'
  q_restate_01     rank 17  found '61,257'
  q_restate_02     rank 15  found '25,048'
  q_split_01       rank  1  found '1.21'
  q_split_02       rank  1  found '24,940'

hybrid / fixed   (searching to depth 50)
  q_numeric_01     rank 12  found '193,737'
  q_numeric_03     rank  6  found '31,376'
  q_numeric_04     rank  8  found '11,350'
  q_temporal_01    rank 26  found '193,737'
  q_restate_01     rank 32  found 

0

**This is the central retrieval finding.** Searching to depth 50:

| Question | dense | BM25 | hybrid |
|---|---|---|---|
| q_numeric_01 (`193,737`) | 12 | 20 | 12 |
| q_split_01 (EPS `1.21`) | **not in top 50** | **1** | 5 |
| q_split_02 (shares `24,940`) | 35 | **1** | 2 |
| q_restate_01 (US revenue) | **not in top 50** | **17** | 32 |

**Dense retrieval cannot find exact figures.** `1.21` is a *token*, not a meaning, and
embeddings encode meaning. BM25 ranks it first; dense cannot find it in fifty results.
This is the concrete case for hybrid retrieval, measured rather than asserted.

## 11 · Measure retrieval · **graded comparison #2**

| Run | Chunking | Retrieval | Isolates |
|---|---|---|---|
| A | fixed | dense | baseline |
| B | SemanticChunker | dense | chunking effect |
| B2 | structural | dense | boundary-source effect |
| C | fixed | hybrid | retrieval method |
| D | fixed | metadata-filtered | `years_present` as a hard filter |
| E | fixed | BM25 only | explains hybrid's result |

Reported **per category**, never only in aggregate — aggregate scores hide where systems
break. And as a **curve over k**, because a single depth conflates "not indexed" with
"ranked too low".

In [22]:
run("evaluate.py", "--runs", "A,B,B2,C,D,E", "--k", "5,10,20")

19 questions, k=[5, 10, 20]

--- k=5 ---
  A   numeric= 37.5% mrr= 37.5 recall=100.0% yr= 81.2% both= 33.3% p50=1150.8ms
  B   numeric= 37.5% mrr= 37.5 recall=100.0% yr= 81.2% both= 33.3% p50= 295.8ms
  B2  numeric= 37.5% mrr= 37.5 recall= 93.8% yr= 78.8% both= 33.3% p50= 214.3ms
  C   numeric= 50.0% mrr= 32.5 recall= 87.5% yr= 66.2% both= 33.3% p50= 258.8ms
  run D FAILED: InternalError: Error executing plan: Internal error: Error finding id
  E   numeric= 25.0% mrr= 21.9 recall=100.0% yr= 65.0% both= 50.0% p50=   8.5ms

--- k=10 ---
  A   numeric= 56.2% mrr= 40.3 recall= 93.8% yr= 68.1% both= 33.3% p50= 201.8ms
  B   numeric= 62.5% mrr= 40.6 recall=100.0% yr= 77.5% both= 33.3% p50= 187.7ms
  B2  numeric= 56.2% mrr= 40.5 recall= 93.8% yr= 69.4% both= 33.3% p50= 187.9ms
  C   numeric= 62.5% mrr= 34.1 recall=100.0% yr= 64.4% both= 66.7% p50= 218.5ms
  run D FAILED: InternalError: Error executing plan: Internal error: Error finding id
  E   numeric= 37.5% mrr= 23.2 recall=100.0% yr= 60.6

0

### What the retrieval scorecard says

**Chunking strategy matters less than retrieval method.** The three chunking arms span
68.8–75.0% numeric accuracy. Swapping dense for metadata-filtered on the *same* chunks
reaches 81.2%.

**Dense fails the split trap completely** — 0% on `split` for every dense arm, 100% for
BM25 and hybrid.

**BM25 surfaces both sides of a restatement 83% of the time** vs dense's 33%, in 2.3ms
vs 200ms.

**Metadata filtering is the cheapest win available:** year precision 70.6% → 90.9% for 13ms.

*Caveat:* n=16 scoreable questions, so one question is 6.25 points. These are directional
findings, not tight measurements.

In [23]:
# Identical aggregates can mean the runs behave identically, OR that they win
# and lose on different questions and the totals coincide. Only a per-question
# view distinguishes those.
run("compare_runs.py")

numeric_hit (Y) and rank of first chunk holding the answer

qid              category            A       B      B2       C       E
----------------------------------------------------------------------
q_numeric_01     numeric        Y   @6  Y   @9  Y   @6  Y   @9  Y  @20
q_numeric_02     numeric        Y   @1  Y   @1  Y   @1  Y   @4  .     
q_numeric_03     numeric        Y   @7  Y   @8  Y   @7  Y   @4  Y  @11
q_numeric_04     numeric        Y  @11  Y   @9  Y  @11  Y   @7  Y  @16
q_temporal_01    temporal       Y  @12  Y   @7  Y  @12  Y  @19  .     
q_temporal_02    temporal       Y   @1  Y   @1  Y   @1  Y  @11  Y  @10
q_temporal_03    temporal       Y   @1  Y   @1  Y   @1  Y   @1  Y   @2
q_trend_01       trend          Y   @7  Y  @11  Y   @8  Y  @11  .     
q_trend_02       trend          .       .       .       .       .     
q_trend_03       trend          Y   @1  Y   @1  Y   @1  Y   @1  .     
q_restate_01     restatement    .       .       .       .       Y  @17
q_restate_02     

0

## 11b · Reranking · **graded comparison #3**

A bi-encoder — the embedding model — encodes the query and each chunk *separately* and
compares vectors. A **cross-encoder** reads the query and chunk *together* and scores the
pair directly. Much more accurate, much more expensive, so it only works as a second pass
over a small candidate pool.

`BAAI/bge-reranker-base` runs locally via `sentence-transformers` — no API call, so the
latency it adds is CPU time rather than network time. That matters because the brief asks
for reranking reported **with timings**, and local timings are stable enough to quote.

**What this has to prove.** By the time it was built, the measurements already showed
hybrid retrieval fixing the split trap dense failed (0% → 100%), and metadata filtering
beating both on numeric accuracy for ~13ms. So a reranker earns its place only if it
improves on *those* — it is scored as runs **F** and **G**, against **C** and **D**, not
against the dense baseline.

In [24]:
run("rerank.py")

loading BAAI/bge-reranker-base ...
loaded in 57.8s

Q: What was NVIDIA's Data Center revenue in fiscal year 2026?

  hybrid alone [  1620ms]
      0.0325 prose  FY2026, Item 7
      0.0323 prose  FY2026, Item 7
      0.0320 prose  FY2024, Item 7

  + rerank s retrieve + 14463ms rerank = 15298ms]
      0.9927 prose  FY2026, Item 7
      0.9758 prose  FY2026, Item 15
      0.9732 prose  FY2026, Item 7

  top-3 changed: True

Q: What was NVIDIA's basic earnings per share in fiscal 2024?

  hybrid alone [   613ms]
      0.0290 prose  FY2025, Item 15
      0.0287 table  FY2024, Item 15, "NVIDIA Corporation and Subsidiaries"
      0.0283 prose  FY2024, Item 15

  + rerank s retrieve + 31978ms rerank = 32558ms]
      0.9600 prose  FY2024, Item 7
      0.7473 prose  FY2024, Item 7
      0.5518 prose  FY2024, Item 15

  top-3 changed: True

Q: What were NVIDIA's United States revenues in fiscal 2025?

  hybrid alone [   583ms]
      0.0294 prose  FY2025, Item 15
      0.0280 prose  FY2026, Item

0

Every top-3 changed, so the cross-encoder is genuinely re-scoring rather than passing the
order through. Note the cost split in the output: ~800ms to retrieve, then **5–12 seconds**
to rerank a pool of 30 short passages.

In [25]:
run("evaluate.py", "--runs", "C,D,F,G", "--k", "20")

Exception in thread Thread-28 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\joshitham_waybetterm\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 1041, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "C:\Users\joshitham_waybetterm\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 992, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\joshitham_waybetterm\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 1611, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "C:\Users\joshitham_waybetterm\AppData\Local\Programs\Python\Python313\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 521: character maps to <undefined>


19 questions, k=[20]

--- k=20 ---
  C   numeric= 75.0% mrr= 33.4 recall=100.0% yr= 67.8% both= 66.7% p50= 209.9ms
  D   numeric= 81.2% mrr= 41.5 recall=100.0% yr= 90.9% both= 33.3% p50= 242.7ms
  F   numeric= 81.2% mrr= 33.1 recall=100.0% yr= 74.7% both= 66.7% p50=9501.8ms
  G   numeric= 75.0% mrr= 28.7 recall=100.0% yr= 92.2% both= 50.0% p50=12381.6ms

RETRIEVAL SCORECARD
run  chunking        retrieval  numeric    mrr  recall  yr_prec   both      p50       p95
-----------------------------------------------------------------------------------------
C    fixed           hybrid       75.0%   33.4  100.0%    67.8%    67%   209.9ms   2384.5ms
D    fixed           filtered     81.2%   41.5  100.0%    90.9%    33%   242.7ms   1990.8ms
F    fixed           rerank       81.2%   33.1  100.0%    74.7%    67%  9501.8ms  18755.9ms
G    fixed           rerank:filtered    75.0%   28.7  100.0%    92.2%    50% 12381.6ms  18865.4ms

NUMERIC ACCURACY BY CATEGORY  (aggregate scores hide where systems b

0

### What reranking actually bought

| Run | Retrieval | numeric | yr_prec | both sides | p50 |
|---|---|---|---|---|---|
| C | hybrid | 75.0% | 67.8% | 67% | **218ms** |
| D | filtered | **81.2%** | **90.9%** | 33% | **245ms** |
| F | **+ rerank over hybrid** | **81.2%** | 74.7% | 67% | 7,257ms |
| G | + rerank over filtered | 75.0% | **92.2%** | 50% | 9,492ms |

**Reranking works — and is dominated by a technique that costs nothing.**

Run F improves on hybrid by 6.2 points of numeric accuracy, fixes temporal (67% → 100%)
and restatement (50% → 75%). That is a real gain, and my prediction that a cross-encoder
would demote tables in favour of prose was **wrong** — it did not hurt numeric accuracy.

But run F lands on **exactly the same 81.2%** as run D, which is metadata filtering — and
D gets there in **245ms against F's 7,257ms**. Same accuracy, **30x** the latency.

**Combining them is worse than either alone.** Run G scores 75.0%, below both its
components, and halves split-trap accuracy (100% → 50%). Filtering narrows the pool to
one fiscal year, so the reranker can no longer surface the *other* filing's version of a
restated figure — the two techniques actively work against each other on exactly the
questions this corpus was built to test.

### The honest conclusion

Reranking is **not** the right answer for this corpus. Not because it fails — it works —
but because a metadata filter extracted at clean time reaches the same accuracy for 3% of
the latency.

That comparison only exists because the reranker was built and measured. Skipping it on
the assumption it would not help would have been the *right call for the wrong reason*,
and unreportable either way.

## 12 · Answers — cited, or refused

Three exits, and the system must choose one:

- **ANSWER** — state the figure as the filing prints it, cite the passage `[n]`
- **CLARIFY** — the question is ambiguous; give the figure for each reading
- **REFUSE** — the passages do not contain it. A confident wrong figure is far worse than
  "I don't know"

Prompts are versioned in `src/prompts.py` (currently **v2**) because the writeup has to
report which prompt produced which numbers.

The v2 change was **the restatement rule**: if the passages hold two different values for
the same line item and period, the answer must report *both* and say which filing each
came from. Reporting one silently is wrong even when the chosen number is correct.

In [26]:
from generate import answer, build_llm

gen_llm = build_llm()
retriever = build("hybrid", "fixed", embeddings)

for q in (
    "What was NVIDIA's Data Center revenue in fiscal year 2026?",            # answers
    "What were NVIDIA's United States revenues in fiscal 2025?",             # restatement
    "What is NVIDIA's projected Data Center revenue for fiscal year 2027?",  # refuses
):
    hits = retriever.search(q, k=20)
    a = answer(q, hits, gen_llm)
    print(f"\n{'=' * 78}\nQ: {q}")
    print(f"exit={a.exit}  citations={a.citations}  {a.latency_ms:.0f}ms")
    print("-" * 78)
    print(a.text[:600])


Q: What was NVIDIA's Data Center revenue in fiscal year 2026?
exit=ANSWER  citations=[12]  936ms
------------------------------------------------------------------------------
Data Center revenue for fiscal year 2026 was $193,737 million [12].



Q: What were NVIDIA's United States revenues in fiscal 2025?
exit=CLARIFY  citations=[1, 9]  1792ms
------------------------------------------------------------------------------
CLARIFY — the question is ambiguous because the passages report U.S. revenue for fiscal 2025 using two different methodologies, resulting in two different figures.

- In the FY2025 filing ([1]), revenue from sales outside the United States was 53% of total revenue, meaning U.S. revenue was **47% of total revenue**. However, this is based on *customer billing location*.

- In the FY2026 filing ([9]), NVIDIA changed its methodology to *customer headquarters location* and restated prior periods accordingly. Under this new methodology, revenue from sales outside the United States was 41% of total 



Q: What is NVIDIA's projected Data Center revenue for fiscal year 2027?
exit=REFUSE  citations=[]  1369ms
------------------------------------------------------------------------------
REFUSE — the passages do not contain the answer.

The question asks for NVIDIA's **projected** Data Center revenue for **fiscal year 2027**. The provided passages contain historical revenue data for fiscal years 2024, 2025, and 2026, but they do not include any forward-looking projections or guidance for fiscal year 2027. 

While some passages discuss growth trends and drivers (e.g., Blackwell architecture, AI demand), none provide a specific figure or estimate for Data Center revenue in FY2027. Forward-looking projections are typically not included in 10-K filings unless accompanied by cautio


> The third question is the important one. Forward-looking guidance is not in a 10-K, so
> the only correct response is to refuse — and to refuse *because the passages lack it*,
> not because the model happens to know NVIDIA does not publish projections.

### Retrieval failure, or generation failure?

When an answer is wrong, it matters which stage caused it — they need opposite fixes.
Running the same question on the retriever's output and on passages guaranteed to contain
the answer separates them.

In [27]:
run("debug_gen.py")


--- hybrid k=10: figure in passages = False ---
exit=REFUSE  citations=[2, 5]
REFUSE â€” the passages do not contain the answer.

While multiple passages discuss Data Center revenue growth in fiscal year 2026 (e.g., up 68% year-on-year [2], with computing up 59% and networking up 142% [5]), none provide the actual dollar amount of Data Ce

--- hybrid k=20: figure in passages = True ---
exit=ANSWER  citations=[12]
Data Center revenue for fiscal year 2026 was $193,737 million [12].

--- oracle passages (1 chunks containing 193,737) ---
    FY2026, Item 15, "The following table summarizes revenue by specialize
exit=ANSWER  citations=[1]
NVIDIA's Data Center revenue in fiscal year 2026 was $193,737 million [1].



0

At k=10 the figure is absent and the system **refuses**. At k=20 it is present and the
system answers `$193,737 million [12]`. Given oracle passages, the same answer.

**The generator never invented a number.** Every failure traced to retrieval depth, not
hallucination.

## 13 · The evaluation table — **the deliverable**

Retrieval scoring asks whether the answer was *retrieved*. This asks whether it was
*answered* — a different question, because a system can retrieve the right passage and
still state the wrong figure, or retrieve nothing and correctly refuse.

| Metric | What it catches |
|---|---|
| exit correct | took the right one of ANSWER / CLARIFY / REFUSE |
| numeric correct | the figure appears in the **answer**, not merely the passages |
| both sides shown | restatement/split: did it state **both** conflicting figures? |
| faithfulness | LLM judge, 0–1, on whether claims trace to passages |
| **hallucination rate** | a figure in the answer that appears in **no** passage |

In [28]:
run("evaluate_gen.py", "--k", "20", "--retriever", "hybrid")

19 questions | hybrid/fixed k=20
judge: on

  [ok ] q_numeric_01     ANSWER   (want ANSWER  ) num=Y faith=1.00   1113ms
  [ok ] q_numeric_02     ANSWER   (want ANSWER  ) num=Y faith=1.00    457ms
  [ok ] q_numeric_03     ANSWER   (want ANSWER  ) num=Y faith=1.00   1403ms
  [ok ] q_numeric_04     ANSWER   (want ANSWER  ) num=Y faith=1.00    664ms
  [   ] q_temporal_01    REFUSE   (want ANSWER  ) num=. faith=1.00   1855ms
  [ok ] q_temporal_02    ANSWER   (want ANSWER  ) num=Y faith=1.00   1367ms
  [ok ] q_temporal_03    CLARIFY  (want CLARIFY ) num=Y faith=1.00   1711ms
  [   ] q_trend_01       ANSWER   (want ANSWER  ) num=. faith=1.00   1097ms
  [   ] q_trend_02       ANSWER   (want ANSWER  ) num=. faith=1.00   1584ms
  [ok ] q_trend_03       ANSWER   (want ANSWER  ) num=Y faith=1.00   3515ms
  [   ] q_restate_01     CLARIFY  (want ANSWER  ) num=. faith=1.00   1703ms
  [   ] q_restate_02     REFUSE   (want ANSWER  ) num=. faith=1.00   1286ms
  [ok ] q_restate_03     ANSWER   (want ANSW

0

**That table is the evaluation deliverable.**

| Metric | Score |
|---|---|
| exit correct | 84.2% |
| numeric correct | 62.5% |
| both sides shown | 50.0% |
| faithfulness | 1.00 |
| **hallucination rate** | **0.0%** |
| invalid citations | 0.0% |

**Zero hallucinations across 19 questions.** No answer ever stated a figure that was not
in its passages, and refusal was 100% correct on the unanswerable category. The system
fails by declining to answer, never by inventing.

The weak spot is `restatement` — 50% exit, 25% numeric. That is the hardest category and
the one the corpus was chosen to expose: it requires noticing that two filings disagree,
retrieving *both*, and saying so.


## 14 · The LangGraph loop

```
retrieve ──> grade ──(sufficient)──> generate ──> verify ──> END
             │                                      │
             │(insufficient, once)                  │(ungrounded)
             ↓                                      ↓
          rewrite ────────> retrieve             refuse ──> END
```

**Built last, deliberately.** If every measurement above had run through a loop with a
retry node, a score improving would be ambiguous: better chunking, or the retry happening
to fire? The linear path produced the numbers first; this graph is measured *against*
them.

Why each node earns its place:

- **grade** — the measured failure mode was retrieval *depth*, not hallucination. At k=10
  the answer was absent and the system refused; at k=20 it answered correctly. A grader
  that notices "this question wants a figure and no passage has one" can trigger one
  deeper retry instead of refusing a question the corpus can answer.
- **rewrite** — financial queries fail lexically in a predictable way: a user writes
  "earnings per share", the filing says "net income per share". One rewrite toward the
  filing's vocabulary is cheap and targeted.
- **verify** — citations are checked against the passages actually supplied, so a citation
  to a passage that was never provided is caught.

Retry is capped at **one**. An uncapped loop can spend unbounded credits on a question the
corpus genuinely cannot answer — and `unanswerable` is a category this eval set
deliberately contains.

In [29]:
run("graph.py")

prompt version: v2

Q: What was NVIDIA's Data Center revenue in fiscal year 2026?
   . retrieve(k=10, query="What was NVIDIA's Data Center revenue in fiscal ") -> 10 hits
   . grade: wants_number=True has_table=False has_digits=False -> insufficient
   . rewrite: k->20, terms=['revenue']
   . retrieve(k=20, query="What was NVIDIA's Data Center revenue total reve") -> 20 hits
   . grade: wants_number=True has_table=True has_digits=True -> sufficient
   . generate: exit=ANSWER citations=[3]
   . verify: all figures and citations sourced
   exit=ANSWER  rewrites=1  grounded=True
------------------------------------------------------------------------------
NVIDIA's Data Center revenue in fiscal year 2026 was $193,737 million [3].

This figure is from the table in Item 15 of the FY2026 filing, which reports Data Center revenue for the year ended January 25, 2026, as $193,737 million.

Q: What was NVIDIA's basic earnings per share in fiscal 2024?
   . retrieve(k=10, query="What was NVIDIA's

0

Read the traces above:

- **Q1** the grader found no figures at k=10, rewrote, retrieved at k=20, answered
  `$193,737 million [3]`. This is the exact question that failed in the linear path.
- **Q2** sufficient at k=10 — answered with **both** split figures (`$12.05` pre-split,
  `$1.21` post-split) and named the filings. The restatement rule firing.
- **Q3** rewrote, retrieved deeper, still correctly refused. The loop does not talk
  itself into answering an unanswerable question.

### Does the loop earn its latency?

A retry loop costs latency and credits, so it has to be measured rather than assumed to
help. Same eval set, same retriever, compared against the linear scorecard.

In [30]:
run("evaluate_graph.py")

19 questions through the graph (hybrid/fixed, k=10 then 20 on retry)

  [ok ] q_numeric_01     ANSWER   (want ANSWER  ) num=Y rewrites=1   7214ms
  [ok ] q_numeric_02     ANSWER   (want ANSWER  ) num=Y rewrites=0   2594ms
  [ok ] q_numeric_03     ANSWER   (want ANSWER  ) num=Y rewrites=0   1242ms
  [ok ] q_numeric_04     ANSWER   (want ANSWER  ) num=Y rewrites=0   1043ms
  [   ] q_temporal_01    ANSWER   (want ANSWER  ) num=. rewrites=0   2480ms
  [ok ] q_temporal_02    ANSWER   (want ANSWER  ) num=Y rewrites=0   1457ms
  [ok ] q_temporal_03    CLARIFY  (want CLARIFY ) num=Y rewrites=1   7585ms
  [ok ] q_trend_01       ANSWER   (want ANSWER  ) num=Y rewrites=1   3353ms
  [   ] q_trend_02       ANSWER   (want ANSWER  ) num=. rewrites=1   2881ms
  [ok ] q_trend_03       ANSWER   (want ANSWER  ) num=Y rewrites=0   4075ms
  [   ] q_restate_01     CLARIFY  (want ANSWER  ) num=. rewrites=1   8793ms
  [ok ] q_restate_02     ANSWER   (want ANSWER  ) num=Y rewrites=1   4186ms
  [   ] q_restate_

0

### What the graph earns its latency question comes down to

| Metric | Linear | Graph | Delta |
|---|---|---|---|
| exit correct | 89.5% | 84.2% | −5.3 |
| numeric correct | 62.5% | **81.2%** | **+18.8** |
| both sides shown | 50.0% | **66.7%** | **+16.7** |
| faithfulness | 1.00 | 1.00 | 0.0 |
| hallucination rate | 0.0% | 0.0% | 0.0 |
| p50 latency | 3,301ms | 6,884ms | **+3,583** |

**The loop is a real quality win at roughly 2x latency.** Numeric accuracy 62.5% → 81.2%
comes almost entirely from grade-and-retry: 8 of 19 questions triggered one rewrite, and
the k=10 → k=20 escalation closes exactly the gap the depth curve identified. Restatement
numeric accuracy went 25% → 75%.

**Whether that trade is worth taking depends on the latency ceiling** — a first-class
constraint in the brief that this project never pinned down. At a 5-second budget the
graph does not qualify; at 20 seconds it clearly does. Reporting both rather than picking
one is the honest move.

**The verify node never fired.** Across 19 questions the generator never produced an
unsourced figure or an invalid citation, so the safety net caught nothing. Good for the
system, and an honest finding about the node: on this corpus it is insurance, not a
contributor. It would matter more with a weaker generator or a noisier corpus.

### A bug worth showing

The graph's *first* scorecard looked contradictory: numeric accuracy **up** 18.8 points,
exit correctness **down** 21.1. Answers were finding the right figure and then being
labelled wrong.

The exit classifier was keyword-matching instead of reading the model's own declaration.
An answer opening `"CLARIFY — the question is ambiguous"` was classified REFUSE because it
later said one reading "cannot be determined from these passages".

Fixed by reading the declared exit first, with one asymmetric override: a declared REFUSE
whose body states figures *with citations* is treated as an ANSWER. The six cases below
are taken verbatim from real answers rather than invented — and one of them initially
"failed" because the *test* was wrong, not the code.

In [31]:
run("test_classify.py")

  [ok  ] clarify-with-refusal-aside         got=CLARIFY  want=CLARIFY
  [ok  ] answer-with-however                got=ANSWER   want=ANSWER
  [ok  ] mislabelled-refuse-that-answers    got=ANSWER   want=ANSWER
  [ok  ] genuine-refusal                    got=REFUSE   want=REFUSE
  [ok  ] plain-answer                       got=ANSWER   want=ANSWER
  [ok  ] declared-answer                    got=ANSWER   want=ANSWER

all 6 cases classified correctly



0